# Sales & Revenue Analysis
**Objective:** Explore the Superstore-style sales dataset, clean it, derive KPIs, and visualise key business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load the Dataset

In [ ]:
df = pd.read_csv('../data/sales_data.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Data Cleaning

In [ ]:
# Check for nulls
print('Missing values:\n', df.isnull().sum())

# Convert Order Date to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'\nRemoved {before - len(df)} duplicate rows')

# Validate numeric columns
df['Sales']    = pd.to_numeric(df['Sales'],    errors='coerce')
df['Profit']   = pd.to_numeric(df['Profit'],   errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df.dropna(subset=['Sales', 'Profit', 'Quantity'], inplace=True)

# Add helper columns
df['Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['Year']  = df['Order Date'].dt.year

print(f'\nClean dataset shape: {df.shape}')
df.dtypes

## 3. KPI Metrics

In [ ]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
total_qty     = df['Quantity'].sum()
avg_sales     = df['Sales'].mean()
profit_margin = (total_profit / total_sales) * 100

print(f'Total Sales      : ${total_sales:,.2f}')
print(f'Total Profit     : ${total_profit:,.2f}')
print(f'Total Qty Sold   : {int(total_qty):,}')
print(f'Average Sales    : ${avg_sales:,.2f}')
print(f'Profit Margin    : {profit_margin:.2f}%')

## 4. Sales Trend Over Time

In [ ]:
monthly_sales = df.groupby('Month')['Sales'].sum().reset_index().sort_values('Month')

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales['Month'], monthly_sales['Sales'], marker='o', color='#2d6a9f', linewidth=2)
plt.title('Monthly Sales Trend', fontsize=16, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Revenue by Category

In [ ]:
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=cat_sales.index, y=cat_sales.values, palette='Set2')
plt.title('Revenue by Category', fontsize=16, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Total Sales ($)')
for i, v in enumerate(cat_sales.values):
    plt.text(i, v + 500, f'${v:,.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(cat_sales)

## 6. Top 5 Products by Sales

In [ ]:
top5 = df.groupby('Product Name')['Sales'].sum().nlargest(5).sort_values()

plt.figure(figsize=(10, 5))
top5.plot(kind='barh', color='#2d6a9f')
plt.title('Top 5 Products by Sales', fontsize=16, fontweight='bold')
plt.xlabel('Total Sales ($)')
plt.ylabel('')
for i, v in enumerate(top5.values):
    plt.text(v + 200, i, f'${v:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 7. Regional Sales Analysis

In [ ]:
region_sales = df.groupby('Region')['Sales'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(region_sales.values, labels=region_sales.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel'), startangle=140)
axes[0].set_title('Regional Sales Share', fontsize=14, fontweight='bold')

# Bar chart
sns.barplot(x=region_sales.index, y=region_sales.values, palette='Set1', ax=axes[1])
axes[1].set_title('Sales by Region', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Total Sales ($)')

plt.tight_layout()
plt.show()

## 8. Profit Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Profit by Category
profit_cat = df.groupby('Category')['Profit'].sum()
colors_cat = ['#2ecc71' if v > 0 else '#e74c3c' for v in profit_cat.values]
axes[0].bar(profit_cat.index, profit_cat.values, color=colors_cat)
axes[0].set_title('Profit by Category', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Total Profit ($)')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

# Profit by Region
profit_reg = df.groupby('Region')['Profit'].sum()
colors_reg = ['#2ecc71' if v > 0 else '#e74c3c' for v in profit_reg.values]
axes[1].bar(profit_reg.index, profit_reg.values, color=colors_reg)
axes[1].set_title('Profit by Region', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Total Profit ($)')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()

## 9. Business Insights Summary

In [ ]:
best_category  = cat_sales.idxmax()
best_product   = df.groupby('Product Name')['Sales'].sum().idxmax()
best_region    = region_sales.idxmax()
most_profit    = profit_reg.idxmax()
peak_month     = monthly_sales.loc[monthly_sales['Sales'].idxmax(), 'Month']

print('===== BUSINESS INSIGHTS =====')
print(f'Highest Revenue Category : {best_category}')
print(f'Best Selling Product      : {best_product}')
print(f'Most Profitable Region    : {most_profit}')
print(f'Highest Sales Region      : {best_region}')
print(f'Peak Sales Month          : {peak_month}')